**Figure 3: Overhead Scaling with Trajectory Length.** Experiments run the
deterministic coding workload at 54/64/96 calls (2 repeats). (a) per-call latency;
(b) end-to-end wall time; (c) isolation cost relative to bare, with the red dotted
line marking parity; (d) the read-trace penalty in isolation. Results suggest that
per-call overhead grows only mildly with trajectory length (2.6x to 3.1x over bare)
-- the tax is per-boundary, not super-linear -- so the motivation holds at scale and
the read-trace multiplier (~2x) stays the dominant term.


In [ ]:
# ipython -c "%run plot_scaling.ipynb"
# Shared USENIX plotting convention (FAST/OSDI camera-ready).
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import style
import pandas as pd
import numpy as np
from pathlib import Path

STANDARD_WIDTH = 17.8            # USENIX two-column text width, in cm
SINGLE_COL_WIDTH = STANDARD_WIDTH / 2
DOUBLE_COL_WIDTH = STANDARD_WIDTH

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
style.use('bmh')
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['hatch.linewidth'] = 0.5
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['font.family'] = 'Nimbus Roman'
pd.options.display.max_columns = None

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

plt.rcParams['axes.grid.axis'] = 'both'
df = pd.read_csv(RESULTS / 'motivation_scaling.csv')
lengths = sorted(df['length'].unique())
modes = ['bare', 'agenttx_without_read_tracing', 'agenttx_full']
labels = ['Bare', 'AgentTX no-trace', 'AgentTX full']
colors = ['#2b2d42', '#8d99ae', '#ef233c']
markers = ['x', 'o', '>']
line_types = ['--', '-', '-']

def series(metric, mode):
    return [float(df[(df['length'] == length) & (df['mode'] == mode)][metric].iloc[0]) for length in lengths]

per_step = {mode: series('per_step_mean_ms', mode) for mode in modes}
wall = {mode: series('wall_mean_s', mode) for mode in modes}
bare = np.asarray(per_step['bare'])
no_trace = np.asarray(per_step['agenttx_without_read_tracing'])
full = np.asarray(per_step['agenttx_full'])

fig = plt.figure(dpi=300, figsize=(cm_to_inch(DOUBLE_COL_WIDTH), cm_to_inch(5.8)))
plots = [
    ('per_step_mean_ms', 'Per-call latency (ms)', '(a) Per-call overhead', per_step, '{:.0f}'),
    ('wall_mean_s', 'Trajectory latency (s)', '(b) End-to-end workload', wall, '{:.1f}'),
    ('overhead', 'Overhead vs. bare (x)', '(c) Isolation cost', {'agenttx_without_read_tracing': (no_trace / bare).tolist(), 'agenttx_full': (full / bare).tolist()}, '{:.1f}x'),
    ('trace_penalty', 'Full / no-trace (x)', '(d) Read-trace penalty', {'agenttx_full': (full / no_trace).tolist()}, '{:.2f}x'),
]
line_handles = []
for plot_id, (metric, ylabel, title, values, formatter) in enumerate(plots):
    ax = plt.subplot(2, 2, plot_id + 1)
    plot_modes = list(values)
    for mode_id, mode in enumerate(plot_modes):
        mode_idx = modes.index(mode)
        line, = ax.plot(lengths, values[mode], color=colors[mode_idx], marker=markers[mode_idx], linestyle=line_types[mode_idx], linewidth=0.7, markersize=3, markeredgewidth=0.6, label=labels[mode_idx])
        if plot_id == 0:
            line_handles.append(line)
        for x_value, y_value in zip(lengths, values[mode]):
            offset = 4 if (mode_idx != 2 or plot_id == 3) else -9
            ax.annotate(formatter.format(y_value), (x_value, y_value), textcoords='offset points', xytext=(0, offset), ha='center', fontsize=5)
    if metric == 'overhead':
        ax.axhline(1.0, color=colors[2], linewidth=0.7, linestyle=':', zorder=1)
    ax.set_title(title, fontsize=8)
    ax.set_xlabel('Trajectory length (# calls)', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xticks(lengths)
    ax.tick_params(bottom=False, top=False, left=False, right=False)
    ax.tick_params(axis='both', labelsize=8)
    for axis in ['top', 'bottom', 'left', 'right']:
        ax.spines[axis].set_linewidth(0.5)

fig.legend(line_handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.03), ncol=3, frameon=False, columnspacing=1.0, handletextpad=0.2, handlelength=1.5, fontsize=8)
plt.tight_layout(pad=0.4, rect=[0.04, 0.0, 0.99, 0.92])
plt.savefig(FIGDIR / 'FIG-Motivation-Scaling.pdf', bbox_inches='tight', pad_inches=0)
plt.savefig(FIGDIR / 'FIG-Motivation-Scaling.png', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()

print(f"isolation cost (full vs bare) across lengths: {[f'{v:.2f}x' for v in (full / bare)]}")
print(f"read-trace penalty across lengths:            {[f'{v:.2f}x' for v in (full / no_trace)]}")
